# Credit Balance Analysis

This notebook analyses credit card balance data, examining how balance varies across three binary categorical variables:
- **Married** (Yes / No)
- **Student** (Yes / No)
- **Own house** (Yes / No)

For each variable we produce:
1. Summary statistics
2. Boxplot + density histogram
3. Pairplot
4. Welch two-sample t-test

## 1. Import Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette('husl')
plt.rcParams['font.size']   = 12
plt.rcParams['font.family'] = 'sans-serif'

## 2. Load & Inspect Data

In [2]:
credit = pd.read_csv("https://raw.githubusercontent.com/kostis-christodoulou/data_analytics_executives/main/data/credit.csv")


# Rename 'own' to 'own_house' for clarity
credit = credit.rename(columns={'own': 'own_house'})

print(f"Shape: {credit.shape}")
credit.info()

Shape: (400, 11)
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   income     400 non-null    float64
 1   limit      400 non-null    int64  
 2   rating     400 non-null    int64  
 3   cards      400 non-null    int64  
 4   age        400 non-null    int64  
 5   education  400 non-null    int64  
 6   own_house  400 non-null    str    
 7   student    400 non-null    str    
 8   married    400 non-null    str    
 9   region     400 non-null    str    
 10  balance    400 non-null    int64  
dtypes: float64(1), int64(6), str(4)
memory usage: 34.5 KB


In [ ]:
credit.head()

## 3. Summary Statistics

Overall balance distribution, then broken down by each grouping variable.

In [ ]:
print("--- Overall balance ---")
display(credit['balance'].describe().round(2).to_frame().T)

print("\n--- Balance by married ---")
display(credit.groupby('married')['balance'].describe().round(2))

print("\n--- Balance by student ---")
display(credit.groupby('student')['balance'].describe().round(2))

print("\n--- Balance by own_house ---")
display(credit.groupby('own_house')['balance'].describe().round(2))

## 4. Helper: Plot + T-Test

A reusable function that produces the boxplot, density histogram, pairplot, and Welch t-test for any binary grouping variable vs balance.

In [ ]:
def analyse_balance_by_group(df, group_col, group_label):
    """
    For a binary categorical variable:
      1. Boxplot + density histogram side by side
      2. Pairplot (balance vs group)
      3. Welch two-sample t-test with R-style summary output
    """
    groups = df[group_col].dropna().unique()

    # --- Plot 1: Boxplot + Density histogram ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle(f'Balance vs. {group_label}', fontsize=14, fontweight='bold')

    # Boxplot: median, IQR, whiskers, and outlier dots per group
    sns.boxplot(data=df, x='balance', y=group_col, ax=ax1)
    ax1.set_title('Boxplot')
    ax1.set_xlabel('Balance')
    ax1.set_ylabel('')

    # Overlapping density histograms: density=True normalises both groups
    # so they are comparable even if group sizes differ
    for g in groups:
        subset = df[df[group_col] == g]['balance'].dropna()
        ax2.hist(subset, alpha=0.3, density=True, label=g, bins=20)
    ax2.set_xlabel('Balance')
    ax2.set_ylabel('Density')
    ax2.set_title('Density Histogram')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    # --- Plot 2: Pairplot ---
    # Shows the joint and marginal distributions for the two variables
    g = sns.pairplot(df[[group_col, 'balance']], hue=group_col, diag_kind='hist')
    g.fig.suptitle(f'Pairplot: Balance vs. {group_label}', y=1.02)
    plt.show()

    # --- Welch two-sample t-test ---
    # equal_var=False: Welch's test, does not assume equal variance between groups
    # This is the safer default when group sizes or spreads may differ
    x = df[df[group_col] == groups[0]]['balance'].dropna()
    y = df[df[group_col] == groups[1]]['balance'].dropna()

    t_stat, p_value = stats.ttest_ind(x, y, equal_var=False)

    # Compute Welch df and 95% CI for the difference in means
    diff = x.mean() - y.mean()
    se   = np.sqrt(x.var(ddof=1)/len(x) + y.var(ddof=1)/len(y))
    df_w = (x.var()/len(x) + y.var()/len(y))**2 / (
           (x.var()/len(x))**2/(len(x)-1) + (y.var()/len(y))**2/(len(y)-1))
    ci   = stats.t.interval(0.95, df=df_w, loc=diff, scale=se)
    sig  = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'

    print(f"--- Welch T-Test: balance by {group_label} ---")
    print(f"  n         : {len(x)} ({groups[0]})  vs  {len(y)} ({groups[1]})")
    print(f"  mean      : {x.mean():.2f}  vs  {y.mean():.2f}")
    print(f"  std       : {x.std(ddof=1):.2f}  vs  {y.std(ddof=1):.2f}")
    print(f"  diff      : {diff:.2f}  ({groups[0]} minus {groups[1]})")
    print(f"  95% CI    : [{ci[0]:.2f}, {ci[1]:.2f}]")
    print(f"  t         : {t_stat:.4f}")
    print(f"  df        : {df_w:.2f}")
    print(f"  p-value   : {p_value:.4f}  {sig}")
    print()

print("Helper function defined.")

## 5. Balance vs. Married

In [ ]:
analyse_balance_by_group(credit, 'married', 'Married')

## 6. Balance vs. Student

In [ ]:
analyse_balance_by_group(credit, 'student', 'Student')

## 7. Balance vs. Own House

In [ ]:
analyse_balance_by_group(credit, 'own_house', 'Own House')

## 8. Summary of T-Test Results

A quick side-by-side comparison of all three tests.

In [ ]:
results = []
for col, label in [('married', 'Married'), ('student', 'Student'), ('own_house', 'Own House')]:
    groups = credit[col].dropna().unique()
    x = credit[credit[col] == groups[0]]['balance'].dropna()
    y = credit[credit[col] == groups[1]]['balance'].dropna()
    t_stat, p_value = stats.ttest_ind(x, y, equal_var=False)
    results.append({
        'Variable':  label,
        f'Mean ({groups[0]})': round(x.mean(), 2),
        f'Mean ({groups[1]})': round(y.mean(), 2),
        'Difference': round(x.mean() - y.mean(), 2),
        't-stat':    round(t_stat, 4),
        'p-value':   round(p_value, 4),
        'Sig':       '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
    })

display(pd.DataFrame(results).set_index('Variable'))